_IN6227 Assignment 1 — Variant 1: Comparing Two Classifiers_

Course assignment: IN6227-Assignment-1

This notebook is the reproducible companion to the two-column PDF report. It proceeds from data inspection and preprocessing to validation-based tuning, final evaluation and discussion. All analysis uses the supplied `train.csv` and `test.csv` files.

How to run: install `requirements.txt`; place `train.csv` and `test.csv` in a local `dataset/` folder, or set the `IN6227_DATA_DIR` environment variable to their folder; run cells from top to bottom. The data are intentionally not published in the GitHub repository. The notebook's documented results were independently produced by `assignment1_analysis.py` with random seed 42; running all cells should reproduce them up to minor library-version differences.

_1. Goal and evaluation plan_

The target is binary (`yes` / `no`). I compare logistic regression with a random forest because the former gives a simple linear reference model and the latter can model nonlinear interactions. The comparison is more informative than evaluating only one model.

I use the supplied training file for a stratified 80/20 development split. Candidate settings are chosen only on the validation part. A five-fold stratified check of the same 12 settings then describes how sensitive that choice is to the split; it does not replace the original selection. After selecting one setting per model family, I refit on all labeled training rows and evaluate the supplied test file. Average precision (AP) is the primary selection metric because only about 24% of rows are `yes`; accuracy alone can hide poor detection of the less common class. I also report accuracy, balanced accuracy, precision, recall, F1, ROC-AUC and confusion matrices so the trade-offs are visible. This expanded grid and five-fold check were requested after an earlier test evaluation had already been viewed, so the test data are no longer a fresh, untouched confirmation of the extended analysis.

In [1]:
import os
import hashlib
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
DATA_DIR = Path(os.environ.get("IN6227_DATA_DIR", "dataset"))
if not (DATA_DIR / "train.csv").exists() and (DATA_DIR / "dataset" / "train.csv").exists():
    DATA_DIR = DATA_DIR / "dataset"
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
assert train.columns.tolist() == test.columns.tolist(), "Train and test schemas differ"
assert "label" in train.columns
feature_columns = [c for c in train.columns if c != "label"]
numeric = train[feature_columns].select_dtypes(include="number").columns.tolist()
categorical = [c for c in feature_columns if c not in numeric]
print(f"Raw train/test rows: {len(train):,} / {len(test):,}")
print(f"Predictors: {len(feature_columns)} ({len(numeric)} numeric, {len(categorical)} categorical)")
print("Numeric:", numeric)
print("Categorical:", categorical)
print(f"Environment: Python {sys.version.split()[0]}, NumPy {np.__version__}, pandas {pd.__version__}, scikit-learn {sklearn.__version__}")
for source in ("train.csv", "test.csv"):
    digest = hashlib.sha256((DATA_DIR / source).read_bytes()).hexdigest()
    print(f"{source} SHA-256: {digest}")

Raw train/test rows: 31,112 / 13,334
Predictors: 15 (7 numeric, 8 categorical)
Numeric: ['aptitude_score', 'performance_score', 'stability_index', 'load_ratio', 'index_weight', 'activity_duration', 'composite_rank']
Categorical: ['geological_era', 'weather_pattern', 'region', 'personal_interest', 'brew_preference', 'instrument', 'species', 'mineral_type']
Environment: Python 3.10.8, NumPy 2.2.6, pandas 2.3.3, scikit-learn 1.7.2
train.csv SHA-256: 5211257faf4212e248fb5661b3bfa1a639bfb4c54a646619c4d906a3e1cfa0a2
test.csv SHA-256: eee9de74f74357ad72848b36e8c34f09ad60ea2b60ea7ebd346be987f4909ac8


_2. Data inspection and cleaning decisions_

The data have 31,112 training rows, 13,334 test rows and 15 predictors. Three training rows and one test row lack the target label. They cannot contribute to supervised fitting or scoring, so I exclude just those four rows. For missing predictor values, I keep the rows and impute within each training fold; dropping complete rows would discard information unnecessarily. I keep the provided train/test files separate throughout model selection.

The files contain no exact duplicate rows within either file. I therefore do not apply arbitrary deduplication. Numeric columns have long tails (for example `load_ratio`), but without feature definitions I cannot establish that extreme values are errors. I retain them rather than silently clipping values. The same caution applies to rare categorical levels. I do not create synthetic features: the meaning and collection timing of these variables are unspecified, and engineered combinations could be hard to justify or unavailable when making predictions.

In [2]:
train_clean = train.dropna(subset=["label"]).copy()
test_clean = test.dropna(subset=["label"]).copy()
assert set(train_clean["label"]) == {"no", "yes"}
assert set(test_clean["label"]) == {"no", "yes"}

summary = pd.DataFrame({
    "train": [len(train), len(train_clean), int(train[feature_columns].isna().sum().sum()),
              int(train.duplicated().sum()), float((train_clean["label"] == "yes").mean())],
    "test": [len(test), len(test_clean), int(test[feature_columns].isna().sum().sum()),
             int(test.duplicated().sum()), float((test_clean["label"] == "yes").mean())],
}, index=["raw rows", "labeled rows", "missing predictor cells", "exact duplicate rows", "yes prevalence"])
display(summary)
display(train[feature_columns].isna().sum().loc[lambda s: s > 0].to_frame("missing train cells"))
display(train[numeric].describe(percentiles=[0.25, 0.5, 0.75]).T[["min", "25%", "50%", "75%", "max"]])

,train,test
raw rows,31112.000000,13334.000000
labeled rows,31109.000000,13333.000000
missing predictor cells,47.000000,20.000000
exact duplicate rows,0.000000,0.000000
yes prevalence,0.239931,0.237606


,missing train cells
geological_era,2
weather_pattern,3
region,4
aptitude_score,4
performance_score,2
stability_index,2
personal_interest,6
load_ratio,2
brew_preference,7
instrument,3


,min,25%,50%,75%,max
aptitude_score,90.000,310.5975,333.515,402.56250,510.000
performance_score,0.000,146.5725,275.385,420.17000,1010.000
stability_index,0.000,2.7510,3.351,3.76175,4.625
load_ratio,0.000,19.2720,23.689,27.65675,143.717
index_weight,0.000,7.5200,11.730,16.23000,99.620
activity_duration,0.000,72.7400,80.330,90.25750,205.000
composite_rank,0.589,2.5910,3.199,3.83800,6.876


_3. Features, split and leakage-safe preprocessing_

I retain all 15 supplied predictors (7 numeric and 8 categorical). Logistic regression needs comparable numeric scales, so numeric missing values receive the training-fold median and numeric values are standardized. The median is less affected by long tails than the mean. Categorical missing values receive the training-fold mode, then categories are one-hot encoded. Unknown categories in validation or test are ignored rather than causing a failure. These transformations are inside a scikit-learn `Pipeline`; `fit` learns imputation values, scales and category lists only from the appropriate training data. Scaling is unnecessary for trees but harmless and allows a consistent preprocessing design.

The split is stratified because the positive class is less common. Seed 42 makes it repeatable. I also report five-fold training-only AP to show how the one-split parameter choice varies across partitions; this later diagnostic is not used to revise the test evaluation.

In [3]:
# pandas nullable string columns are converted to object so sklearn receives np.nan.
for frame in (train_clean, test_clean):
    frame[categorical] = frame[categorical].astype(object)
    frame[categorical] = frame[categorical].where(pd.notna(frame[categorical]), np.nan)

X = train_clean[feature_columns]
y = (train_clean["label"] == "yes").astype(int)
X_test = test_clean[feature_columns]
y_test = (test_clean["label"] == "yes").astype(int)
X_fit, X_val, y_fit, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

def preprocessing():
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric),
        ("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical),
    ])

print(f"Development fit/validation rows: {len(X_fit):,} / {len(X_val):,}")
print(f"Positive prevalence: fit={y_fit.mean():.3%}, validation={y_val.mean():.3%}")

Development fit/validation rows: 24,887 / 6,222
Positive prevalence: fit=23.992%, validation=23.995%


_4. Model configuration and recorded parameter search_

I compare a declared 12-setting grid on the same stratified validation split. Logistic regression uses L2 regularization with `C = 0.1, 1, 10` (larger `C` means weaker regularization) and `class_weight = None` or `balanced`. Balanced weights test whether giving the less frequent `yes` class more influence improves ranking; they also change the scores at the fixed 0.50 threshold. `lbfgs` has a 1,000-iteration cap. The random forest fixes 150 trees and compares `max_depth = 12` or unrestricted with `min_samples_leaf = 2, 5, 10`. A larger leaf can reduce variance but may underfit. The forest completes after 150 trees; each tree stops when a split is not permitted by its depth/leaf constraints or no improvement is possible. Each family is selected by its highest validation average precision (AP), with the test labels excluded.

Scores at threshold 0.50 are supplementary. I do not tune that threshold on the test set. If the real application has a specified cost for missed `yes` cases, it should choose a threshold on validation data before test evaluation.

In [4]:
def summarize(y_true, probability):
    predicted = probability >= 0.5
    return {
        "accuracy": accuracy_score(y_true, predicted),
        "balanced_accuracy": balanced_accuracy_score(y_true, predicted),
        "precision_yes": precision_score(y_true, predicted, zero_division=0),
        "recall_yes": recall_score(y_true, predicted, zero_division=0),
        "f1_yes": f1_score(y_true, predicted, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probability),
        "average_precision": average_precision_score(y_true, probability),
        "confusion_matrix_no_yes": confusion_matrix(y_true, predicted).tolist(),
    }

candidates = {}
for weight in (None, "balanced"):
    for c in (0.1, 1, 10):
        candidates[f"logistic_C_{c:g}_weight_{weight or 'none'}"] = LogisticRegression(
            C=c, class_weight=weight, max_iter=1000, solver="lbfgs", random_state=SEED
        )
for depth in (12, None):
    for leaf in (2, 5, 10):
        candidates[f"forest_depth_{depth or 'none'}_leaf_{leaf}"] = RandomForestClassifier(
            n_estimators=150, max_depth=depth, min_samples_leaf=leaf, n_jobs=-1, random_state=SEED
        )
validation = {}
for name, classifier in candidates.items():
    pipe = Pipeline([("prep", preprocessing()), ("classifier", classifier)])
    pipe.fit(X_fit, y_fit)
    validation[name] = summarize(y_val, pipe.predict_proba(X_val)[:, 1])
    if name.startswith("logistic"):
        validation[name]["iterations"] = int(pipe.named_steps["classifier"].n_iter_[0])

validation_table = pd.DataFrame(validation).T.drop(columns="confusion_matrix_no_yes")
display(validation_table[["average_precision", "roc_auc", "recall_yes", "iterations"]].round(5))
selections = {
    family: max((name for name in candidates if name.startswith(prefix)),
                key=lambda name: validation[name]["average_precision"])
    for family, prefix in [("logistic_regression", "logistic"),
                           ("random_forest", "forest")]
}
print("Selected by validation AP:", selections)

,average_precision,roc_auc,recall_yes,iterations
logistic_C_0.1_weight_none,0.696288,0.885553,0.551239,75
logistic_C_1_weight_none,0.697381,0.886337,0.559946,87
logistic_C_10_weight_none,0.697749,0.88652,0.561956,145
logistic_C_0.1_weight_balanced,0.692895,0.884936,0.851306,77
logistic_C_1_weight_balanced,0.694289,0.885609,0.851306,95
logistic_C_10_weight_balanced,0.69451,0.885781,0.853985,124
forest_depth_12_leaf_2,0.696411,0.883728,0.496986,NaN
forest_depth_12_leaf_5,0.693075,0.88396,0.496316,NaN
forest_depth_12_leaf_10,0.693138,0.883618,0.488948,NaN
forest_depth_none_leaf_2,0.690368,0.883446,0.539853,NaN


Selected by validation AP: {'logistic_regression': 'logistic_C_10_weight_none', 'random_forest': 'forest_depth_none_leaf_5'}


Recorded validation finding (seed 42): Without class weights, logistic AP rises from 0.69629 at `C=0.1` to 0.69775 at `C=10`; `C=10` wins by AP and converges in 145 iterations, below the 1,000-iteration cap. Balanced class weights increase positive recall at threshold 0.50 to about 0.85 but lower AP to 0.69289-0.69451. This shows that weighting changes the threshold trade-off, not that it improves ranking. For the forest, unrestricted depth with leaf size 5 has the highest validation AP (0.69663); depth 12 with leaf size 2 is close (0.69641). More restrictive leaves or a too-small leaf with unrestricted trees can lose AP on this split. These small differences are empirical observations, not evidence that one setting is reliably superior across other splits. The full 12-row search is recorded above.

_Five-fold sensitivity check of the same 12 settings_

The following diagnostic repeats the search across five stratified folds of the labeled training file. Every fold fits imputation, scaling, encoding and the model on its own training portion. I report mean and sample standard deviation of fold AP, along with mean ROC-AUC and positive recall at 0.50. Because these settings were examined after an earlier test inspection, and the original 80/20 validation subset is contained within these folds, this is a stability check rather than an independent model selection or a new test estimate. The original selected settings remain fixed for the supplied test evaluation.

In [5]:
cv_rows = []
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for name, classifier in candidates.items():
    for fold, (fit_index, val_index) in enumerate(folds.split(X, y), start=1):
        pipe = Pipeline([("prep", preprocessing()), ("classifier", classifier)])
        pipe.fit(X.iloc[fit_index], y.iloc[fit_index])
        truth = y.iloc[val_index]
        probability = pipe.predict_proba(X.iloc[val_index])[:, 1]
        cv_rows.append({
            "candidate": name, "fold": fold,
            "average_precision": average_precision_score(truth, probability),
            "roc_auc": roc_auc_score(truth, probability),
            "recall_at_0_5": recall_score(truth, probability >= 0.5),
        })
cv_folds = pd.DataFrame(cv_rows)
cv_summary = cv_folds.groupby("candidate").agg(
    cv_ap_mean=("average_precision", "mean"),
    cv_ap_sd=("average_precision", "std"),
    cv_roc_auc_mean=("roc_auc", "mean"),
    cv_recall_mean=("recall_at_0_5", "mean"),
).sort_values("cv_ap_mean", ascending=False)
display(cv_summary.round(5))

,cv_ap_mean,cv_ap_sd,cv_roc_auc_mean,cv_recall_mean
candidate,,,,
forest_depth_none_leaf_10,0.71467,0.01850,0.89095,0.54515
forest_depth_12_leaf_2,0.71411,0.01860,0.89020,0.51461
forest_depth_12_leaf_5,0.71371,0.01829,0.89023,0.50764
forest_depth_none_leaf_5,0.71359,0.01891,0.89051,0.55507
forest_depth_12_leaf_10,0.71303,0.01836,0.89011,0.50657
logistic_C_10_weight_none,0.70929,0.01540,0.89062,0.56431
logistic_C_1_weight_none,0.70904,0.01545,0.89066,0.56243
forest_depth_none_leaf_2,0.70838,0.01684,0.88839,0.56284
logistic_C_0.1_weight_none,0.70799,0.01545,0.89046,0.55681


The five-fold mean AP is 0.70929 (SD 0.01540) for the selected logistic setting, and 0.71359 (SD 0.01891) for the selected forest setting. The highest forest mean is 0.71467 with unlimited depth and leaf size 10, only 0.00108 above the originally selected leaf size 5. This small reversal is a warning that the forest's one-split optimum is unstable; it is not a reason to switch models after viewing test results. The comparison of two models on the supplied test set therefore stays descriptive.

_Feature-availability sensitivity: `composite_rank`_

The name `composite_rank` suggests a derived feature, but the data provide no provenance or collection time. If it is unavailable when predictions are made, using it would be inappropriate. I therefore repeat the already selected model configurations on the same validation split after removing this column. This is a sensitivity check, not another opportunity to tune the test result. The ablation does not establish whether the remaining predictors would be available in deployment.

In [6]:
without_rank = [c for c in feature_columns if c != "composite_rank"]
numeric_without = [c for c in numeric if c != "composite_rank"]
categorical_without = [c for c in categorical if c != "composite_rank"]
preprocessor_without_rank = ColumnTransformer([
    ("numeric", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_without),
    ("categorical", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_without),
])
sensitivity_rows = []
for family, selected_name in selections.items():
    pipe = Pipeline([("prep", preprocessor_without_rank),
                     ("classifier", candidates[selected_name])])
    pipe.fit(X_fit[without_rank], y_fit)
    probability = pipe.predict_proba(X_val[without_rank])[:, 1]
    sensitivity_rows.append({
        "model": family,
        "AP all 15": validation[selected_name]["average_precision"],
        "AP without composite_rank": average_precision_score(y_val, probability),
    })
sensitivity_table = pd.DataFrame(sensitivity_rows).set_index("model")
sensitivity_table["AP change"] = (sensitivity_table["AP without composite_rank"]
                                  - sensitivity_table["AP all 15"])
display(sensitivity_table.round(5))

,AP all 15,AP without composite_rank,AP change
model,,,
logistic_regression,0.69775,0.69769,-0.00006
random_forest,0.69663,0.69618,-0.00044


Interpretation: In the verified run, removing `composite_rank` changed validation AP by less than 0.0005 for either model. That small change suggests the observed comparison is not driven by this one feature on this split. It does not prove the feature is available at prediction time, nor that the result is stable in new data.

_Validation-only threshold trade-off_

Average precision describes ranking across thresholds, while a decision requires a threshold and an error-cost policy. The 0.50 threshold is held fixed for the reported test results. To make its trade-off visible, I recompute the two selected models on the original development fit portion and inspect four illustrative thresholds on the validation portion only. No threshold is selected from this table for the supplied test evaluation. A deployment threshold would need a stated false-negative cost or recall target and a fresh evaluation.

In [7]:
threshold_rows = []
for family, selected_name in selections.items():
    pipe = Pipeline([("prep", preprocessing()),
                     ("classifier", candidates[selected_name])])
    pipe.fit(X_fit, y_fit)
    probability = pipe.predict_proba(X_val)[:, 1]
    for threshold in (0.30, 0.40, 0.50, 0.60):
        predicted = probability >= threshold
        tn, fp, fn, tp = confusion_matrix(y_val, predicted).ravel()
        threshold_rows.append({
            "model": family, "threshold": threshold,
            "precision_yes": precision_score(y_val, predicted, zero_division=0),
            "recall_yes": recall_score(y_val, predicted, zero_division=0),
            "f1_yes": f1_score(y_val, predicted, zero_division=0),
            "false_positives": int(fp), "false_negatives": int(fn),
        })
threshold_table = pd.DataFrame(threshold_rows).set_index(["model", "threshold"])
display(threshold_table.round(4))

precision_yes  recall_yes  f1_yes  false_positives  false_negatives
model               threshold                                                                     
logistic_regression 0.3               0.5703      0.7930  0.6635              892              309
                    0.4               0.6323      0.6772  0.6539              588              482
                    0.5               0.6843      0.5620  0.6171              387              654
                    0.6               0.7483      0.4461  0.5590              224              827
random_forest       0.3               0.5651      0.7964  0.6611              915              304
                    0.4               0.6272      0.6772  0.6512              601              482
                    0.5               0.6852      0.5291  0.5971              363              703
                    0.6               0.7433      0.4113  0.5295              212              879

For logistic regression on this validation split, moving from 0.50 to 0.30 raises positive recall from 0.5620 to 0.7930 and reduces false negatives from 654 to 309, while false positives rise from 387 to 892. The forest shows a similar pattern. This illustrates the cost trade-off; it is not a recommendation to deploy 0.30. The reported test results continue to use the fixed 0.50 threshold.

_5. Refit and evaluate the supplied test file_

Once settings are frozen, I refit each pipeline on all labeled training rows. The supplied test file is scored at a fixed 0.50 threshold. The reported AP and ROC-AUC use probabilities and therefore do not depend on this threshold. The `no`-only baseline achieves about 76.2% test accuracy while detecting zero `yes` cases; that is why accuracy by itself is insufficient. Because this is a follow-up search after an earlier test inspection, the test comparison is descriptive rather than a newly independent confirmation.

In [8]:
held_out = {}
test_probabilities = {}
for family, selected_name in selections.items():
    pipe = Pipeline([("prep", preprocessing()),
                     ("classifier", candidates[selected_name])])
    pipe.fit(X, y)
    probability = pipe.predict_proba(X_test)[:, 1]
    test_probabilities[family] = probability
    held_out[family] = summarize(y_test, probability)

test_table = pd.DataFrame(held_out).T.drop(columns="confusion_matrix_no_yes")
display(test_table.round(4))
print(f"Always-no accuracy: {(y_test == 0).mean():.4f}; yes recall: 0")
for family, result in held_out.items():
    print(f"{family} confusion matrix (true rows / predicted columns: no, yes):")
    display(pd.DataFrame(result["confusion_matrix_no_yes"],
                         index=["true no", "true yes"],
                         columns=["pred no", "pred yes"]))

,accuracy,balanced_accuracy,precision_yes,recall_yes,f1_yes,roc_auc,average_precision
logistic_regression,0.839571,0.747253,0.698572,0.571338,0.628581,0.888825,0.702472
random_forest,0.839271,0.745101,0.700274,0.565657,0.625808,0.889711,0.708431


Always-no accuracy: 0.7624; yes recall: 0
logistic_regression confusion matrix (true rows / predicted columns: no, yes):


,pred no,pred yes
true no,9384,781
true yes,1358,1810


random_forest confusion matrix (true rows / predicted columns: no, yes):


,pred no,pred yes
true no,9398,767
true yes,1376,1792


Recorded test finding: Logistic regression: accuracy 0.8396, positive recall 0.5713, F1 0.6286, AP 0.7025. Random forest: accuracy 0.8393, positive recall 0.5657, F1 0.6258, AP 0.7084. Both have nearly identical accuracy. Logistic regression finds 18 more positive cases at threshold 0.50 (1,810 versus 1,792), while the forest has slightly higher AP, which evaluates ranking across thresholds. Both miss more than 1,350 of the 3,168 actual positive cases; the operational cost of those misses matters more than a tiny accuracy difference.

_6. Uncertainty in the AP difference_

The forest-minus-logistic AP gap is about 0.006. I use 1,000 paired bootstrap resamples of test rows to describe its sampling uncertainty: each resample uses the same selected rows for both models, preserving the paired comparison. This interval is descriptive, not a new model-selection step, and it does not capture uncertainty from selecting hyperparameters on one validation split. The original test observations remain unchanged.

In [9]:
rng = np.random.default_rng(SEED)
y_array = y_test.to_numpy()
paired_differences = []
for _ in range(1000):
    sample = rng.integers(0, len(y_array), len(y_array))
    if len(np.unique(y_array[sample])) < 2:
        continue
    paired_differences.append(
        average_precision_score(y_array[sample], test_probabilities["random_forest"][sample])
        - average_precision_score(y_array[sample], test_probabilities["logistic_regression"][sample])
    )
interval = np.quantile(paired_differences, [0.025, 0.975])
gap = held_out["random_forest"]["average_precision"] - held_out["logistic_regression"]["average_precision"]
print(f"Forest - logistic test AP: {gap:.4f}")
print(f"Paired bootstrap 95% interval: [{interval[0]:.4f}, {interval[1]:.4f}]")

Forest - logistic test AP: 0.0060
Paired bootstrap 95% interval: [-0.0015, 0.0132]


Interpretation: The recorded interval is approximately [-0.0015, 0.0132]. It includes zero, so these data do not identify a clear AP winner. This does not prove the models are equivalent; it means the observed advantage is too small to claim confidently from this comparison.

_7. Discussion, limitations and decision_

1. Which model would I use? With no stated error costs, neither is a decisive winner. Logistic regression is a sensible starting choice because it is simpler and has slightly better positive recall/F1 at threshold 0.50. The forest has a small AP advantage that might matter if ranking candidates is the application. A final choice requires the application's false-positive/false-negative costs and verified feature availability.
2. Post-hoc extension. The original test set was viewed before this requested expansion. The 12 candidates are compared and selected using validation AP alone, and the winning configurations match those from the earlier analysis. Even so, a fresh external test set would be needed for an independent confirmation of the expanded workflow.
3. Validation sensitivity. Five-fold AP keeps the logistic choice but changes the forest's highest-mean leaf size from 5 to 10 by only 0.00108. The fold SD for the selected forest is 0.01891. The original single-split winner is therefore fragile, and the five-fold check does not provide an independent confirmation after the earlier test inspection.
4. Unknown provenance. Feature names do not establish how or when predictors were collected. For example, a derived field such as `composite_rank` may be unavailable at prediction time; deployment would need a provenance check.
5. Dataset-specific findings. The reported scores describe the supplied dataset and split; performance on another dataset or at another time has not been measured.

Conclusion: Both models reach about 84% accuracy on the supplied test file, but the minority-class recall is only about 57%. The small AP difference does not support declaring a winner. The reproducible validation-to-test workflow and clear treatment of the less common `yes` class are the main findings.